In [2]:
!pip install --upgrade qunetsim

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.5/60.5 kB 3.8 MB/s eta 0:00:00
  Created wheel for eqsn: filename=EQSN-0.0.8-py3-none-any.whl size=11940 sha256=17a6e36491cd44a913294641fabaec3e79eb9e56636aa8cd1c3a5c4acc48c75c
  Stored in directory: /root/.cache/pip/wheels/09/72/a2/791e035cfdb1c48129a58d1dfa0667d1b3befb7481c1ff0155
Successfully built eqsn


In [3]:
import numpy as np
import random
from qunetsim.components import Host, Network
from qunetsim.objects import Qubit, Logger

# Disable verbose logging
Logger.DISABLED = True

wait_time = 10  # seconds to wait for qubits

In [4]:
# XOR encryption scheme used only for demonstration
# MUST NOT be used for real cryptography.
def encrypt(key, text):
    encrypted_text = ""
    for char in text:
        encrypted_text += chr(ord(key) ^ ord(char))
    return encrypted_text

def decrypt(key, encrypted_text):
    return encrypt(key, encrypted_text)

def key_array_to_key_string(key_array):
    key_string_binary = ''.join([str(x) for x in key_array])
    return ''.join(chr(int(''.join(x), 2)) for x in zip(*[iter(key_string_binary)] * 8))

In [5]:
def alice_qkd(alice, msg_buff, secret_key, receiver):
    sequence_nr = 0
    for bit in secret_key:
        ack = False
        while not ack:
            print(f"Alice sent {sequence_nr + 1} key bits")
            base = random.randint(0, 1)  # 0 = Z-basis, 1 = X-basis
            q_bit = Qubit(alice)
            if bit == 1:
                q_bit.X()
            if base == 1:
                q_bit.H()
            alice.send_qubit(receiver, q_bit, await_ack=True)
            message = alice.get_classical(receiver, wait=wait_time, seq_num=sequence_nr)
            if message is not None:
                message = message.content  # extract the actual string content
            if message == f"{sequence_nr} : {base}":
                ack = True
                alice.send_classical(receiver, f"{sequence_nr} :0", await_ack=True)
            else:
                ack = False
                alice.send_classical(receiver, f"{sequence_nr} :1", await_ack=True)
        sequence_nr += 1

In [6]:
def eve_qkd(eve, msg_buff, key_size, sender):
    sequence_nr = 0
    received_counter = 0
    key_array = []
    while received_counter < key_size:
        measurement_base = random.randint(0, 1)
        q_bit = eve.get_qubit(sender, wait=wait_time)
        while q_bit is None:
            q_bit = eve.get_qubit(sender, wait=wait_time)
        if measurement_base == 1:
            q_bit.H()
        bit = q_bit.measure()
        eve.send_classical(sender, f"{sequence_nr} : {measurement_base}", await_ack=True)
        msg_obj = eve.get_classical(sender, wait=wait_time, seq_num=sequence_nr)
        if msg_obj is not None:
            msg = msg_obj.content
        else:
            msg = None
        if msg == f"{sequence_nr} :0":
            received_counter += 1
            print(f"Eve received {received_counter} key bits.")
            key_array.append(bit)
        sequence_nr += 1
    return key_array

In [7]:
def alice_send_message(alice, secret_key, receiver):
    msg_to_eve = "Hi Eve, how are you???"
    secret_key_string = key_array_to_key_string(secret_key)
    encrypted_msg_to_eve = encrypt(secret_key_string, msg_to_eve)
    print("Alice sends encrypted message")
    alice.send_classical(receiver, f"-1:{encrypted_msg_to_eve}", await_ack=True)

def eve_receive_message(eve, msg_buff, eve_key, sender):
    # For the final message we use seq_num=-1 (as per example)
    msg_obj = eve.get_classical(sender, wait=wait_time, seq_num=-1)
    if msg_obj is not None:
        encrypted_msg_from_alice = msg_obj.content.split(':', 1)[1]
        secret_key_string = key_array_to_key_string(eve_key)
        decrypted_msg_from_alice = decrypt(secret_key_string, encrypted_msg_from_alice)
        print(f"Eve received decoded message: {decrypted_msg_from_alice}")
    else:
        print("Eve did not receive the encrypted message.")

In [ ]:
def main():
    network = Network.get_instance()
    nodes = ['Alice', 'Bob', 'Eve']
    network.delay = 0.0
    network.start(nodes)

    host_alice = Host('Alice')
    host_alice.add_connection('Bob')
    host_alice.start()

    host_bob = Host('Bob')
    host_bob.add_connection('Alice')
    host_bob.add_connection('Eve')
    host_bob.start()

    host_eve = Host('Eve')
    host_eve.add_connection('Bob')
    host_eve.start()

    network.add_host(host_alice)
    network.add_host(host_bob)
    network.add_host(host_eve)

    key_size = 10
    secret_key = np.random.randint(2, size=key_size)

    def alice_func(alice):
        msg_buff = []
        alice_qkd(alice, msg_buff, secret_key, host_eve.host_id)
        alice_send_message(alice, secret_key, host_eve.host_id)

    def eve_func(eve):
        msg_buff = []
        eve_key = eve_qkd(eve, msg_buff, key_size, host_alice.host_id)
        eve_receive_message(eve, msg_buff, eve_key, host_alice.host_id)

    t1 = host_alice.run_protocol(alice_func, ())
    t2 = host_eve.run_protocol(eve_func, ())
    t1.join()
    t2.join()

    network.stop(True)

if __name__ == '__main__':
    main()

Alice sent 1 key bits
Eve received 1 key bits.
Alice sent 2 key bits
Eve received 2 key bits.
Alice sent 3 key bits
Alice sent 3 key bits
Alice sent 4 key bits
Alice sent 4 key bits
Alice sent 5 key bits
Alice sent 5 key bits
Alice sent 5 key bits
Alice sent 6 key bits
Alice sent 6 key bits
Alice sent 7 key bits
Alice sent 7 key bits
Alice sent 8 key bits
Alice sent 8 key bits
Alice sent 8 key bits
Alice sent 9 key bits
Alice sent 10 key bits
Alice sent 10 key bits
Alice sent 10 key bits
Alice sent 10 key bits
Alice sent 10 key bits
Alice sends encrypted message
